In [37]:
from pathlib import Path
import os
import sys
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
from datetime import datetime, timedelta
import json
import uuid
import random 
from sqlalchemy import create_engine, func
from notebooks.utils import reset_db, get_session, model_to_dict
from data.models import cultpass

# Udahub Accounts

## Cultpass Database

**Init DB**

In [38]:
cultpass_db = "data/external/cultpass.db"

In [39]:
reset_db(cultpass_db)

✅ Removed existing data/external/cultpass.db
2025-12-03 13:48:47,070 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-12-03 13:48:47,072 INFO sqlalchemy.engine.Engine COMMIT
✅ Recreated data/external/cultpass.db with fresh schema


In [40]:
engine = create_engine(f"sqlite:///{cultpass_db}", echo=False)
cultpass.Base.metadata.create_all(engine)

**Experiences**

In [41]:
experience_data = []

with open('data/external/cultpass_experiences.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        experience_data.append(json.loads(line))

In [42]:
experience_data

[{'title': 'Carnival History Tour in Olinda',
  'description': "Discover the origins and vibrant traditions of Pernambuco's Carnival.",
  'location': 'Pernambuco, Brazil'},
 {'title': 'Sunset Paddleboarding',
  'description': 'Glide across calm waters at golden hour with all gear included.',
  'location': 'Santa Catarina, Brazil'},
 {'title': 'Pelourinho Colonial Walk',
  'description': 'Wander through colorful streets and learn about Afro-Brazilian history.',
  'location': 'Bahia, Brazil'},
 {'title': 'Samba Night at Lapa',
  'description': 'Dance the night away at a traditional samba club in the Lapa arches.',
  'location': 'Rio de Janeiro, Brazil'},
 {'title': 'Christ the Redeemer Experience',
  'description': 'Take a guided trip to one of the New Seven Wonders of the World with historical context.',
  'location': 'Rio de Janeiro, Brazil'},
 {'title': 'Modern Art at MASP',
  'description': 'Enjoy a guided visit to the São Paulo Museum of Art with insights into its top collections.',

In [43]:
with get_session(engine) as session:
    experiences = []

    for idx, experience in enumerate(experience_data):
        exp = cultpass.Experience(
            experience_id=str(uuid.uuid4())[:6],
            title=experience["title"],
            description=experience["description"],
            location=experience["location"],
            when=datetime.now() + timedelta(days=idx+1),
            slots_available=random.randint(1,30),
            is_premium=(idx % 2 == 0)
        )
        experiences.append(exp)

    session.add_all(experiences)

**User**

In [44]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [45]:
cultpass_users

[{'id': 'a4ab87',
  'name': 'Alice Kingsley',
  'email': 'alice.kingsley@wonderland.com',
  'is_blocked': True},
 {'id': 'f556c0',
  'name': 'Bob Stone',
  'email': 'bob.stone@granite.com',
  'is_blocked': False},
 {'id': '88382b',
  'name': 'Cathy Bloom',
  'email': 'cathy.bloom@florals.org',
  'is_blocked': False},
 {'id': '888fb2',
  'name': 'David Noir',
  'email': 'david.noir@shadowmail.com',
  'is_blocked': True},
 {'id': 'f1f10d',
  'name': 'Eva Green',
  'email': 'eva.green@ecosoul.net',
  'is_blocked': False},
 {'id': 'e6376d',
  'name': 'Frank Ocean',
  'email': 'frank.ocean@seawaves.io',
  'is_blocked': False}]

In [46]:
with get_session(engine) as session:
    db_users = []
    for user_info in cultpass_users:
        user = cultpass.User(
            user_id=user_info["id"],
            full_name=user_info["name"],
            email=user_info["email"],
            is_blocked=user_info["is_blocked"],
            created_at=datetime.now()
        )
        db_users.append(user)
    session.add_all(db_users) 

**Subscription**

In [47]:
with get_session(engine) as session:
    subscriptions = []
    for user_info in cultpass_users:
        subscription = cultpass.Subscription(
            subscription_id=str(uuid.uuid4())[:6],
            user_id=user_info["id"],
            status=random.choice(["active", "cancelled"]),
            tier=random.choice(["basic", "premium"]),
            monthly_quota=random.randint(2,10),
            started_at=datetime.now()
        )
        subscriptions.append(subscription)

    session.add_all(subscriptions)

**Reservation**

In [48]:
# Applicable to `cultpass_users[0]` at the moment

with get_session(engine) as session:
    experience_ids = [
        exp.experience_id 
        for exp 
        in session.query(cultpass.Experience).all()
    ]

    reservation1 = cultpass.Reservation(
        reservation_id=str(uuid.uuid4())[:6],
        user_id=cultpass_users[0]["id"],
        experience_id=random.choice(experience_ids),
        status="reserved",
    )

    reservation2 = cultpass.Reservation(
        reservation_id=str(uuid.uuid4())[:6],
        user_id=cultpass_users[0]["id"],
        experience_id=random.choice(experience_ids),
        status="reserved",
    )

    session.add_all([reservation1, reservation2])

In [49]:
# Add more Reservations
with get_session(engine) as session:
    experience_ids = [
        exp.experience_id 
        for exp in session.query(cultpass.Experience).all()
    ]

    print(f"Found {len(cultpass_users)} users and {len(experience_ids)} experiences")

    new_reservations = []

    # For each user, create 2 random reservations
    for user in cultpass_users:
        for _ in range(2):
            new_reservations.append(
                cultpass.Reservation(
                    reservation_id=str(uuid.uuid4())[:6],
                    user_id=user["id"],
                    experience_id=random.choice(experience_ids),
                    status="reserved",   
                )
            )

    session.add_all(new_reservations)

print(f"Created {len(new_reservations)} new reservations for all users.")


Found 6 users and 7 experiences
Created 12 new reservations for all users.


# Tests

In [50]:
with get_session(engine) as session:
    users = session.query(cultpass.User).all()
    for user in users:
        print(user)

<User(user_id='a4ab87', email='alice.kingsley@wonderland.com', is_blocked=True)>
<User(user_id='f556c0', email='bob.stone@granite.com', is_blocked=False)>
<User(user_id='88382b', email='cathy.bloom@florals.org', is_blocked=False)>
<User(user_id='888fb2', email='david.noir@shadowmail.com', is_blocked=True)>
<User(user_id='f1f10d', email='eva.green@ecosoul.net', is_blocked=False)>
<User(user_id='e6376d', email='frank.ocean@seawaves.io', is_blocked=False)>


In [51]:
with get_session(engine) as session:
    users = session.query(cultpass.User).all()
    for user in users:
        print(user.subscription)

<Subscription(subscription_id='4e26b0', user_id='a4ab87', status='cancelled', tier='premium')>
<Subscription(subscription_id='279e9c', user_id='f556c0', status='active', tier='basic')>
<Subscription(subscription_id='342f45', user_id='88382b', status='cancelled', tier='basic')>
<Subscription(subscription_id='aacaf3', user_id='888fb2', status='active', tier='premium')>
<Subscription(subscription_id='b67e3d', user_id='f1f10d', status='active', tier='premium')>
<Subscription(subscription_id='356343', user_id='e6376d', status='active', tier='premium')>


In [52]:
with get_session(engine) as session:
    experiences = session.query(cultpass.Experience).all()
    for experience in experiences:
        print(experience)

<Experience(experience_id='2b165e', title='Carnival History Tour in Olinda', when='2025-12-04 13:48:51.677287')>
<Experience(experience_id='11d80f', title='Sunset Paddleboarding', when='2025-12-05 13:48:51.677392')>
<Experience(experience_id='e36e14', title='Pelourinho Colonial Walk', when='2025-12-06 13:48:51.677440')>
<Experience(experience_id='8f7d4e', title='Samba Night at Lapa', when='2025-12-07 13:48:51.677490')>
<Experience(experience_id='9d3802', title='Christ the Redeemer Experience', when='2025-12-08 13:48:51.677529')>
<Experience(experience_id='c7423b', title='Modern Art at MASP', when='2025-12-09 13:48:51.677569')>
<Experience(experience_id='938e76', title='Ibirapuera Park Bike Ride', when='2025-12-10 13:48:51.677607')>


In [53]:
with get_session(engine) as session:
    count = session.query(func.count(cultpass.Reservation.reservation_id)).scalar()
    print("Total reservations:", count)

    # Show sample entries
    print("\nSample reservations:")
    for r in session.query(cultpass.Reservation).limit(10).all():
        print(f"{r.reservation_id} | user={r.user_id} | exp={r.experience_id} | {r.status}")

Total reservations: 14

Sample reservations:
b1e509 | user=a4ab87 | exp=c7423b | reserved
ed36d8 | user=a4ab87 | exp=2b165e | reserved
23e1c1 | user=a4ab87 | exp=9d3802 | reserved
aadd4c | user=a4ab87 | exp=9d3802 | reserved
ae24e5 | user=f556c0 | exp=8f7d4e | reserved
d9d387 | user=f556c0 | exp=8f7d4e | reserved
47c8a3 | user=88382b | exp=11d80f | reserved
24008a | user=88382b | exp=9d3802 | reserved
ec8b56 | user=888fb2 | exp=2b165e | reserved
ddccb9 | user=888fb2 | exp=8f7d4e | reserved
